Describing Welfare
==================

**Author:** Ethan Ligon



## Introduction



### Reading



### Measuring Individual Welfare



### Material Well-Being: Statistics



### Conventional options for $w$



### Conventional options for $w$, continued



## The Theoretical Object



### The household's problem



### The marginal utility of expenditure



### The martingale condition



### Why total expenditure?



### Hall's random walk



### The assumptions, collected



## Practical Difficulties in Measuring Individual Items of Consumption



### The four components of consumption expenditures



### List Length



### Timing & Recall



### Food prices: which price?



### Food prices: what GLSS7 asks, and what it gets



### Food quantities, and what was never bought



### Building the aggregate



`LSMS_Library` does the food side for you.  Start from what was acquired:



In [1]:
# Show tracebacks without the library's internal frames: the line that
# failed, and why.  Change Plain to Verbose if you ever want the rest.
%xmode Plain

# The first call that builds a table may print "DVC unavailable ...
# falling back to manual aggregation".  That is about how the library
# fetches its raw files, not about your data; the numbers are the same,
# and it does not recur once the table is built.

# These notebooks are read two ways: on your own screen, and projected onto
# a wall in a lit room where thin lines and pale markers are simply not
# there.  Those want different plots, so the difference is a switch rather
# than a compromise that suits neither.  Leave it False; the lecturer flips
# it.  Everything below the `if' is about the wall, not about your laptop.
PROJECTOR = False

import matplotlib as mpl
mpl.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'lines.linewidth': 1.8})
if PROJECTOR:
    mpl.rcParams.update({
        'figure.figsize': (7, 4.5),
        'font.size': 15, 'axes.labelsize': 15, 'axes.titlesize': 17,
        'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 13,
        'lines.linewidth': 3.0, 'lines.markersize': 9, 'axes.linewidth': 1.5,
    })

import lsms_library as ll
import numpy as np, pandas as pd

ghana = ll.Country('GhanaLSS')
acq = ghana.food_acquired()
acq.head()

The index is $(t, i, j, u, s, \mathit{visit})$ — wave, household, item,
unit, source, visit — and the columns are `Expenditure`, `Quantity`,
`Price`.  The derived table collapses this to expenditure per household-item:



In [1]:
food = ghana.food_expenditures()
food.head()

Summing over items gives food *purchases* per household.  Not consumption:



In [1]:
purchases = food.groupby(['t', 'i']).sum().squeeze()
purchases.groupby('t').describe().round(0)

`food_expenditures()` returns what households *bought*.  Food they grew and
ate never had an expenditure attached to it, so it is missing from that
table — missing, not zero.  The acquisition table records it differently:



In [1]:
acq.groupby(['t', 's'])[['Quantity', 'Expenditure', 'Price']].count()

Read that table carefully, because it is the whole problem in one picture.
From 1991-92 on, own production carries a `Quantity` and a `Price` but no
`Expenditure`, while purchases carry a `Quantity` and an `Expenditure` but
no `Price`.  That asymmetry is in the instrument.
[Section 8H](https://hhsurveys.ligonresearch.org/hub/user-redirect/files/reading/GLSS7-2016-17-Section8H-own-produce.pdf) (PDF, opens in a
new tab) asks the quantity of own produce consumed at each of six visits
and, in question 9, one price; the purchase form, Section 9B, further down,
asks amount and quantity and never a price.
The valuation Deaton and Zaidi spend a chapter on is, in this survey, a
multiplication nobody performed:



In [1]:
value = acq.Expenditure.where(acq.Expenditure.notna(), acq.Quantity * acq.Price)
by_source = value.groupby(['t', 's']).sum().unstack('s')
(100 * by_source['produced'] / by_source.sum(axis=1)).round(1)

Own production is 28.7% of the value of food in 2016-17.  So perform the
multiplication.  `value` already has it, expenditure where the household
paid and quantity times its own price where it did not; summed over items
it is food consumption per household, purchased and own-produced, and it
is the aggregate the rest of this session uses.  The library will do the
same for you with `food_expenditures(basis`'total')=; here it is done by
hand so that you can see exactly what was added.



In [1]:
food_total = value.groupby(['t', 'i']).sum()
pd.DataFrame({'purchases': purchases.groupby('t').median(),
              'purchased + own-produced': food_total.groupby('t').median()}).round(0)

The median household's food consumption is well above its food purchases in
every wave, and the gap is what the obvious call would have thrown away.



### GLSS7's answer: a diary, visited six times



### The diary



### GLSS7's six visits



This is the form the diary is kept on, Section 9B: six visit columns headed
*2nd* to *7th*, AMT, QTY and UNIT under each, and nowhere to write a price.
Open it beside the code:
[Section 9B](https://hhsurveys.ligonresearch.org/hub/user-redirect/files/reading/GLSS7-2016-17-Section9B-food-expenditure.pdf) (PDF, opens in a new tab;
also `reading/` in your home directory), and the booklet the household
keeps between visits, which 9B is filled from:
[the diary](https://hhsurveys.ligonresearch.org/hub/user-redirect/files/reading/GLSS7-2016-17-Diary.pdf).
The diary design is visible in the data.  GLSS7 carries a `visit` level:



In [1]:
glss7 = ghana.food_acquired().xs('2016-17', level='t').xs('purchased', level='s')
sorted(glss7.index.get_level_values('visit').unique())

Six of them, five days apart.  The form numbers them 2 to 7, because the
food questions begin at the second visit; the library on the hub labels
them 1 to 6, and other versions use the form's numbers.  The text below uses
the form's numbering, and the code selects the first food visit by position
so that it does not care.  Each visit covers the same five days, so if the
instrument worked perfectly they should all look alike.  They do not:



In [1]:
byvisit = pd.DataFrame({
    'exp':   glss7.groupby(['i', 'visit'])['Expenditure'].sum(),
    'items': glss7.groupby(['i', 'visit']).size(),
})
(byvisit.groupby('visit')
        .agg(mean_exp=('exp', 'mean'), mean_items=('items', 'mean'))
        .assign(exp_per_item=lambda d: d.mean_exp / d.mean_items)
        .round(2))

The first visit stands well above the five that follow, and then the series
is flat.  That is the shape Scott and Amenuvegbe found: steep early, flat
later.  Which of two stories is it, though?  More items remembered, or
bigger amounts attached to the same items?



In [1]:
v1 = sorted(byvisit.index.get_level_values('visit').unique())[0]  # first food visit
first, rest = byvisit.xs(v1, level='visit'), byvisit.drop(v1, level='visit')
for col in ('exp', 'items'):
    print(f"{col:6s} first food visit vs the other five: "
          f"{100 * (first[col].mean() / rest[col].mean() - 1):+.1f}%")

Expenditure is 20.6% higher; the item count only 4.5%.  So it is mostly
larger amounts per item rather than fuller enumeration, which is what
telescoping looks like: purchases from before the window get pulled into
it.  It is the start-up bias Scott and Amenuvegbe had to strip out before
they could measure decay at all, and here it is in the survey you are about
to build an aggregate from.

Whether the first visit is worth keeping is a real question, and
`recall_and_diaries.ipynb` is where you get to argue about it.



### How fast does recall decay?



### What actually drives the error



### Timing: what is a period?



### Inventories: purchases are not consumption



### Price Indices



### A household-specific Paasche index



### Unit values, and what they hide



Look at the unit values directly, for one item, in one round.



In [1]:
uv = (acq.Expenditure / acq.Quantity).rename('unit_value')
uv = uv.replace([np.inf, -np.inf], np.nan).dropna()

item = uv.xs('2016-17', level='t').groupby('j').size().idxmax()  # commonest item
x = uv.xs('2016-17', level='t').xs(item, level='j')
print(item, ':', len(x), 'observations')
x.groupby('u').describe().round(2)          # by unit of measure

Two lessons, both visible in that table.  Unit values can vary enormously within
an item — ratios of 100 to 1 are common, and are misreported quantities,
not real price dispersion.  And they are only comparable *within* a unit of
measure, which is why `u` is in the index.



In [1]:
# The median is the robust summary; the mean is not.
x.groupby('u').agg(['median', 'mean', 'count']).round(2).head()

### Durables and housing



## Intra-Household Allocation



### Adult equivalence



### Barten: children re-price goods



## Constructing Aggregate Welfare Measures



### Poverty measures



### Poverty, correctly weighted



In [1]:
sample = ghana.sample()

def fgt(c, w, z, alpha=0):
    """Weighted FGT_alpha.  c: consumption per adult equivalent; w: weights."""
    c, w = np.asarray(c, float), np.asarray(w, float)
    ok = np.isfinite(c) & np.isfinite(w)
    c, w = c[ok], w[ok]
    # Sum over the poor only.  Writing this as np.sum(w * gap**alpha) over
    # everybody looks equivalent, and is -- except at alpha=0, where
    # 0**0 == 1 and every household in the country counts as poor.
    poor = c < z
    return np.sum(w[poor] * ((z - c[poor]) / z) ** alpha) / np.sum(w)

wave = '2016-17'
c = food_total.xs(wave, level='t')
w = sample.xs(wave, level='t').weight.reindex(c.index)

z = c.quantile(0.25)          # a placeholder line; see the exercise
for a in (0, 1, 2):
    print(f"P_{a} = {fgt(c, w, z, a):.4f}")

# The line is an unweighted quartile, so the UNWEIGHTED headcount must come
# back as 0.25 by construction.  It is the one number here we know in
# advance, which makes it the right thing to check the code against.
print(f"unweighted P_0 = {fgt(c, np.ones(len(c)), z, 0):.4f}")
print(f"mean weight, poor = {w[c < z].mean():.3f}"
      f"   non-poor = {w[c >= z].mean():.3f}")

The unweighted headcount returns 0.2500 — as it must be, since the line
*is* the unweighted lower quartile.  That is the check: it is the one quantity here
whose value you know before running the code, so it is the one that tells
you whether the code is right.  If it comes back as 1.0000, you have found
the $0^0$ trap.

The weighted headcount is 0.2152 — three and a half points lower.  The
reason is in the line below it: poor households carry a mean weight of 0.86
against 1.05 for everyone else, because the design over-sampled them.
You may recall that's Session 1's lesson.

It is worth knowing what these same lines said when `food_total` was
purchases only: a weighted headcount of 0.154 against the same 0.25, and
mean weights of 0.62 for the poor against 1.13.  The gap has narrowed
because the households a purchases-only measure put in the bottom quartile
were, to a large extent, the ones that grow their own food, and those are
also the ones the design over-sampled.  Valued, their own production moves
them up.  On purchases alone, "poor" had largely meant "does not buy".

None of which rescues the line itself.  A quartile of the distribution is
not a poverty line: it fixes the unweighted headcount at 0.25 by
construction, in every country and every year, which is a fact about
arithmetic rather than about Ghana.  A real line is
absolute — the cost of a fixed bundle — and does not move with the
distribution, and does not move when everyone gets poorer.  Fixing that is
the first exercise.



### Inequality



### Inequality: Atkinson



### Lorenz and Gini



In [1]:
def lorenz(c, w):
    c, w = np.asarray(c, float), np.asarray(w, float)
    ok = np.isfinite(c) & np.isfinite(w) & (c >= 0)
    c, w = c[ok], w[ok]
    order = np.argsort(c)
    c, w = c[order], w[order]
    p = np.cumsum(w) / np.sum(w)
    L = np.cumsum(w * c) / np.sum(w * c)
    return np.concatenate([[0], p]), np.concatenate([[0], L])

def gini(c, w):
    p, L = lorenz(c, w)
    return 1 - 2 * np.trapezoid(L, p)

print(f"Gini (food, purchased and own-produced, {wave}) = {gini(c, w):.3f}")

The Atkinson index needs one more thing than the Gini does, and the code
should make you say it out loud: how averse to inequality you are.



In [1]:
def atkinson(c, w, eps):
    """Weighted Atkinson index.  eps > 0 is inequality aversion."""
    c, w = np.asarray(c, float), np.asarray(w, float)
    ok = np.isfinite(c) & np.isfinite(w) & (c > 0)      # log/power need c > 0
    c, w = c[ok], w[ok]
    mean = np.sum(w * c) / np.sum(w)
    if np.isclose(eps, 1):
        ede = np.exp(np.sum(w * np.log(c)) / np.sum(w))
    else:
        ede = (np.sum(w * c ** (1 - eps)) / np.sum(w)) ** (1 / (1 - eps))
    return 1 - ede / mean

for eps in (0.5, 1.0, 2.0):
    print(f"Atkinson A({eps}) = {atkinson(c, w, eps):.3f}")

The library draws the curve too, and does one thing the hand-rolled plot
does not: it writes every choice that moves the curve into the subtitle.



In [1]:
import matplotlib.pyplot as plt

waves = ['1998-99', '2005-06', '2012-13', '2016-17']
if hasattr(ll.visualizations, 'lorenz_curve'):        # development branch only
    ll.visualizations.lorenz_curve(ghana, waves, per='person', basis='total')
else:                                                  # the release: by hand
    fig, ax = plt.subplots(figsize=(5, 5))
    for wv in waves:
        cc = food_total.xs(wv, level='t')
        ww = sample.xs(wv, level='t').weight.reindex(cc.index)
        p, L = lorenz(cc, ww)
        ax.plot(p, L, label=f"{wv}  (G={gini(cc, ww):.2f})")
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5)
    ax.set_xlabel('cumulative share of households')
    ax.set_ylabel('cumulative share of food consumption')
    ax.legend(frameon=False)
plt.show()

The Gini in the legend is not the one you computed above, and the subtitle
says why: it counts *persons*, weighting each household by its size, where
the code above counted households.  Both are Ginis of food consumption in
the same year.  Neither is wrong; the one you report is a choice, and the chart
makes you state it.

If two Lorenz curves cross, the ranking of the two distributions depends on
which inequality measure you chose, and no scalar will rescue you.  Look
before you summarize.



### One index, or several?



### Exercises



1.  The poverty line used above is fake.  Build a real one: take the food
    bundle consumed by households in the second and third deciles, price it at
    national median unit values, and scale it to 2,900 kcal per adult
    equivalent.  Recompute $P_0, P_1, P_2$.
    Notebook: [`poverty_line.ipynb`](poverty_line.ipynb)
2.  Deflate spatially.  Construct a regional Paasche index from the survey's
    own unit values, apply it, and report how much of the north–south poverty
    gradient survives.
    Notebook: [`spatial_deflation.ipynb`](spatial_deflation.ipynb)
3.  Vary $\theta$ in $c_i = C_i / A_i^\theta$ over $[0.5, 1.0]$ and plot
    the headcount ratio against it.  Over what range does the *ranking* of
    regions change?
    Notebook: [`equivalence_scales.ipynb`](equivalence_scales.ipynb)

